<a href="https://colab.research.google.com/github/uwol1116/GenAI-Class/blob/main/Assignment3_2_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 2. Vision Transformer (ViT)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

images, labels = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")


100%|██████████| 170M/170M [02:53<00:00, 982kB/s]


Image batch shape: torch.Size([32, 3, 32, 32])
Label batch shape: torch.Size([32])


## 2-1. Patch Design

In [2]:
PATCH_SIZE  = 4
IMG_SIZE    = 32
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2

print(f"Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"Patch size  : {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Grid        : {IMG_SIZE//PATCH_SIZE} x {IMG_SIZE//PATCH_SIZE}")
print(f"Num patches : {NUM_PATCHES}")


Image size  : 32x32
Patch size  : 4x4
Grid        : 8 x 8
Num patches : 64


32×32 크기의 이미지를 4×4 크기의 패치로 분할하면 가로 방향으로 8개, 세로 방향으로 8개의 패치가 생성되므로 총 64개의 패치가 만들어진다.

각 패치는 NLP에서 단어 토큰이 시퀀스의 기본 단위가 되는 것과 동일하게, ViT에서는 이미지를 구성하는 시각적 토큰으로 처리된다.

패치 크기가 작을수록 더 세밀한 공간 정보를 표현할 수 있지만 패치 수가 늘어나 계산량이 증가하고, 패치 크기가 클수록 계산 효율이 높아지지만 세부 정보가 손실될 수 있기 때문에 적절한 패치 크기 선택이 중요하다.


## 2-2. Patch Embedding

In [3]:
class PatchEmbedding(nn.Module):
    # Conv2d(kernel=patch_size, stride=patch_size)로 패치 추출 + 선형 투영을 한번에 수행
    def __init__(self, img_size=32, patch_size=4, in_ch=3, d_model=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                    # (B, d_model, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)   # (B, num_patches, d_model)
        return x

patch_embed = PatchEmbedding(img_size=32, patch_size=4, in_ch=3, d_model=128)
embedded    = patch_embed(images)

print(f"Input  shape: {images.shape}")
print(f"Output shape: {embedded.shape}")


Input  shape: torch.Size([32, 3, 32, 32])
Output shape: torch.Size([32, 64, 128])


입력 이미지의 shape은 (32, 3, 32, 32)이며, Patch Embedding을 거친 후의 출력 shape은 (32, 64, 128)이다.

첫 번째 차원인 32는 배치 크기를 의미하고, 두 번째 차원인 64는 8×8 격자로 분할된 패치의 수를 의미하며, 세 번째 차원인 128은 각 패치가 선형 투영을 통해 변환된 임베딩 차원(d_model)을 의미한다.

이 과정에서 각 패치는 flatten된 픽셀 벡터로 변환된 후 Linear Projection을 통해 d_model 차원의 벡터로 변환되며, 이후 Transformer의 입력 시퀀스로 사용된다.


## 2-3. Class Token Experiment

In [4]:
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3,
                 d_model=128, nhead=4, num_layers=4,
                 num_classes=10, use_cls_token=True):
        super().__init__()
        self.use_cls     = use_cls_token
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_ch, d_model)
        n_patches        = (img_size // patch_size) ** 2
        seq_len          = n_patches + (1 if use_cls_token else 0)

        if use_cls_token:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.pos_embed   = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(d_model)
        self.head        = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)

        if self.use_cls:
            cls = self.cls_token.expand(B, -1, -1)
            x   = torch.cat([cls, x], dim=1)  # cls token을 시퀀스 앞에 붙임

        x    = x + self.pos_embed
        x    = self.norm(self.transformer(x))
        feat = x[:, 0] if self.use_cls else x.mean(dim=1)  # cls or mean pooling
        return self.head(feat)

vit_with    = ViT(use_cls_token=True)
vit_without = ViT(use_cls_token=False)

vit_with.eval()
vit_without.eval()

with torch.no_grad():
    pred_with    = vit_with(images).argmax(1)[:8]
    pred_without = vit_without(images).argmax(1)[:8]

print(f"Ground Truth         : {labels[:8].tolist()}")
print(f"With CLS token  pred : {pred_with.tolist()}")
print(f"Without CLS token    : {pred_without.tolist()}")


Ground Truth         : [1, 7, 4, 1, 5, 8, 0, 3]
With CLS token  pred : [4, 4, 4, 4, 4, 4, 4, 4]
Without CLS token    : [7, 7, 7, 7, 7, 7, 7, 7]


CLS 토큰을 사용하는 경우, 학습 가능한 벡터인 [CLS] 토큰을 패치 시퀀스의 맨 앞에 추가하고 Transformer Encoder를 통과시킨 후 해당 위치의 출력 벡터를 분류에 사용한다. [CLS] 토큰은 self-attention을 통해 모든 패치 토큰에 접근할 수 있기 때문에, 이미지 전체의 정보를 집약한 global representation을 형성하게 된다. 이는 BERT에서 문장 전체를 하나의 벡터로 표현하기 위해 [CLS] 토큰을 사용하는 것과 동일한 원리이다.

CLS 토큰을 사용하지 않는 경우에는 Transformer를 통과한 모든 패치 토큰의 출력 벡터를 평균하여 분류에 사용한다. 이 방식은 구조가 단순하다는 장점이 있지만, 특정 위치의 토큰이 전체 이미지를 대표하도록 학습되는 것이 아니라 모든 패치의 표현을 동등하게 취급하기 때문에 두 방식의 예측 결과가 서로 다르게 나타난다. 현재 출력된 결과는 훈련 전 랜덤 가중치 상태이므로 정확도 자체는 의미가 없다.


## 2-4. CNN vs ViT Comparison

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cnn = SimpleCNN()

print(f"CNN  parameters : {count_params(cnn):,}")
print(f"ViT  parameters : {count_params(vit_with):,}")

cnn.eval()
with torch.no_grad():
    t0 = time.time()
    for _ in range(30): cnn(images)
    cnn_ms = (time.time() - t0) / 30 * 1000

    t0 = time.time()
    for _ in range(30): vit_with(images)
    vit_ms = (time.time() - t0) / 30 * 1000

print(f"CNN  avg inference : {cnn_ms:.2f} ms")
print(f"ViT  avg inference : {vit_ms:.2f} ms")

with torch.no_grad():
    cnn_preds = cnn(images).argmax(1)
    vit_preds = vit_with(images).argmax(1)

cnn_acc = (cnn_preds == labels).float().mean().item() * 100
vit_acc = (vit_preds == labels).float().mean().item() * 100

print(f"CNN  accuracy (untrained) : {cnn_acc:.1f}%")
print(f"ViT  accuracy (untrained) : {vit_acc:.1f}%")


CNN  parameters : 94,986
ViT  parameters : 546,186
CNN  avg inference : 30.67 ms
ViT  avg inference : 96.10 ms
CNN  accuracy (untrained) : 12.5%
ViT  accuracy (untrained) : 9.4%


CNN은 local receptive field을 기반으로 특징을 추출하기 때문에 translation invariance와 locality라는 inductive bias를 내재적으로 가진다. 이 특성 덕분에 소규모 데이터셋에서도 안정적으로 학습이 가능하며, 파라미터 수가 적고 추론 속도가 빠르다는 장점이 있다.

반면 ViT는 self-attention을 통해 이미지 내 모든 패치 간의 관계를 직접 계산하기 때문에 전역적인 관계를 포착하는 데 유리하다. 그러나 이러한 전역 관계 학습 능력은 충분히 많은 데이터가 주어졌을 때 비로소 발휘되기 때문에, 소규모 데이터셋인 CIFAR-10(32×32)에서는 CNN에 비해 학습 효율이 낮을 수 있다. 파라미터 수와 추론 시간 측면에서도 ViT가 CNN보다 크고 느린 경향이 있다. 대규모 이미지 분류 벤치마크에서는 ViT가 CNN을 능가하는 성능을 보이는 경우가 많기 때문에, 데이터 규모와 태스크 특성에 따라 두 모델 중 적합한 구조를 선택하는 것이 중요하다.
